# Comprehensive AGS benchmark protocol (v2)

**Purpose.** A resumable, reproducible research benchmark for Adaptive Greedy Search (AGS), with SMAC3 as the principal comparator, stronger Optuna TPE variants, random search without replacement, 22 heterogeneous scikit-learn HPO tasks, component ablations, pruning audits, multicore and wall-clock experiments, and task-level statistical analysis.

This notebook is deliberately an **experiment runner**, not a claim that AGS is better. It saves raw per-run records before moving on, so long experiments can be interrupted and resumed. Do not edit results by hand.

## Important methodological decisions

- Every method sees the same outer train/test split and the same inner CV folds for a given task and seed. The held-out test set is not used during search.
- Main comparisons use a common objective-trial/visit budget. Fold-fits and elapsed time are recorded separately because pruning and optimizer overhead differ.
- Optuna is reported in three forms: default TPE, stronger multivariate/group TPE, and deduplicated TPE (duplicate suggestions reuse cached objective values and do not consume model-fit budget).
- SMAC3 uses ConfigSpace's conditional configuration space and a random-forest-based model-based optimizer.
- The 22 default tasks use local scikit-learn datasets and deterministic synthetic datasets, avoiding external data downloads.
- YAHPO Gym is documented in the optional integration cell. Its API is a black-box objective over pre-trained surrogate benchmarks, whereas AGS 3.7.1 is estimator/CV-oriented. Treating repeated folds of a fixed YAHPO function as independent CV folds would be scientifically invalid. A direct objective/fidelity adapter is needed before claiming YAHPO results for AGS.

## Reproducibility warning

The notebook tests the installed package API. Record the exact AGS version and Git commit used. Do not silently upgrade AGS midway through a benchmark. The current repository release is v3.7.1; pin it for the published protocol unless you intentionally benchmark a newer commit.

## How to run

1. Open this notebook from the repository root (or set `REPO_ROOT` below).
2. Run the dependency cell once, then restart the kernel.
3. Run setup and inspect `TASK_CATALOG`. Run the pilot cell first.
4. Run the main benchmark cell. It appends one JSON record per run to `benchmark_artifacts/raw_runs.jsonl`; rerunning skips completed keys.
5. Run the ablation, pruning-audit, scaling/time-budget and statistical-analysis cells.
6. Archive the entire `benchmark_artifacts/` directory with the paper and attach a DOI (e.g. Zenodo) to the exact code/data/results.

**Expected workload:** Full mode evaluates 22 problems × 6 methods × 2 budgets × 10 seeds. Ablations and pruning audits are separate experiments. This is intentionally substantial. Start with `QUICK_MODE=True` to validate API compatibility and data flow.

The benchmark is compute-intensive. Set `N_JOBS` to a sensible number for your machine; avoid nested oversubscription. All underlying estimator fits inside an AGS worker are forced to one job by AGS itself.

In [ ]:
# Optional environment setup. Run once, then restart the kernel.
%pip install -q --upgrade "adaptive-greedy-search==3.7.1" "scikit-learn>=1.4" "optuna>=4.4,<6" "smac>=2.3,<3" "ConfigSpace>=1.2,<2" "scipy>=1.10" "pandas>=2.0" "joblib>=1.3" "matplotlib>=3.7"

# YAHPO Gym is intentionally optional; see the dedicated section below.
# Installation/data setup is documented at:
# https://slds-lmu.github.io/yahpo_gym/getting_started.html

In [ ]:
import os, sys, json, time, math, hashlib, platform, warnings, inspect
from pathlib import Path
from collections import defaultdict
from statistics import mean
from datetime import datetime, timezone
from types import MethodType

import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.datasets import (
    load_breast_cancer, load_digits, load_wine, load_iris,
    load_diabetes, load_linnerud, make_classification, make_moons,
    make_circles, make_regression, make_friedman1, make_friedman2, make_friedman3
)
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier,
    RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
)
from sklearn.svm import SVC, SVR
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, KFold
)
from sklearn.metrics import get_scorer
from joblib import Parallel, delayed

import optuna
from optuna.samplers import TPESampler, RandomSampler
from ags import (
    AdaptiveGreedySearch, NumericParameter, FloatParameter, IntegerParameter,
    OrdinalParameter, CategoricalParameter, ConditionalParameter
)

try:
    import scipy
    from scipy.stats import wilcoxon, friedmanchisquare, rankdata, t as student_t
    HAS_SCIPY = True
except Exception as exc:
    HAS_SCIPY = False
    print("SciPy unavailable:", exc)

try:
    from smac import Scenario, HyperparameterOptimizationFacade
    from ConfigSpace import ConfigurationSpace, CategoricalHyperparameter, EqualsCondition, InCondition
    HAS_SMAC = True
    SMAC_IMPORT_ERROR = None
except Exception as exc:
    HAS_SMAC = False
    SMAC_IMPORT_ERROR = repr(exc)

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=FutureWarning)

# -------------------------- CONFIGURATION --------------------------
QUICK_MODE = False
SEED_BASE = 2026
N_SEEDS = 2 if QUICK_MODE else 10       # use 20 for final paper-grade runs
CV_FOLDS = 5
BUDGETS = [12] if QUICK_MODE else [30, 60]
TEST_SIZE = 0.20
N_JOBS = -1                            # set 1 for serial / -1 for all available cores
ABLATION_BUDGET = 20 if QUICK_MODE else 50
PRUNING_BUDGET = 20 if QUICK_MODE else 50
TIME_BUDGET_SECONDS = 30 if QUICK_MODE else 120
RANDOM_TRIAL_MULTIPLIER = 8            # Optuna deduplication safety cap
RESUME = True
ARTIFACT_DIR = Path("benchmark_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RAW_PATH = ARTIFACT_DIR / "raw_runs.jsonl"
ABLATION_PATH = ARTIFACT_DIR / "ablation_runs.jsonl"
PRUNING_PATH = ARTIFACT_DIR / "pruning_audit.jsonl"
TIMING_PATH = ARTIFACT_DIR / "time_parallel_runs.jsonl"

print("Python:", sys.version.split()[0])
print("AGS:", __import__("ags").__version__)
print("scikit-learn:", sklearn.__version__, "| Optuna:", optuna.__version__)
print("SMAC available:", HAS_SMAC, "| import issue:", SMAC_IMPORT_ERROR)
print("SciPy available:", HAS_SCIPY)
print("CPU count:", os.cpu_count(), "| N_JOBS:", N_JOBS)
print("Artifacts:", ARTIFACT_DIR.resolve())

In [ ]:
# -------------------------- DATASETS --------------------------
def load_problem_data(name, seed=SEED_BASE):
    if name == "breast_cancer":
        d = load_breast_cancer(); return d.data, d.target
    if name == "digits":
        d = load_digits(); return d.data, d.target
    if name == "wine":
        d = load_wine(); return d.data, d.target
    if name == "iris":
        d = load_iris(); return d.data, d.target
    if name == "imbalanced":
        return make_classification(n_samples=900, n_features=35, n_informative=8,
            n_redundant=10, weights=[0.92, 0.08], flip_y=0.02,
            class_sep=0.8, random_state=seed)
    if name == "noisy_classification":
        return make_classification(n_samples=1000, n_features=30, n_informative=8,
            n_redundant=8, n_classes=3, flip_y=0.22, class_sep=0.7,
            random_state=seed)
    if name == "moons":
        return make_moons(n_samples=900, noise=0.28, random_state=seed)
    if name == "circles":
        return make_circles(n_samples=900, noise=0.18, factor=0.35, random_state=seed)
    if name == "highdim_classification":
        return make_classification(n_samples=900, n_features=120, n_informative=18,
            n_redundant=25, n_repeated=5, n_classes=4, class_sep=0.8,
            random_state=seed)
    if name == "smalln_wide":
        return make_classification(n_samples=240, n_features=90, n_informative=12,
            n_redundant=15, n_classes=2, flip_y=0.08, random_state=seed)
    if name == "diabetes":
        d = load_diabetes(); return d.data, d.target
    if name == "linnerud":
        d = load_linnerud(); return d.data, d.target[:, 0]
    if name == "friedman1":
        return make_friedman1(n_samples=900, n_features=20, noise=5.0, random_state=seed)
    if name == "friedman2":
        return make_friedman2(n_samples=900, noise=1.0, random_state=seed)
    if name == "friedman3":
        return make_friedman3(n_samples=900, noise=0.1, random_state=seed)
    if name == "sparse_regression":
        return make_regression(n_samples=900, n_features=80, n_informative=8,
            noise=18.0, random_state=seed)
    if name == "nonlinear_regression":
        X, y = make_friedman1(n_samples=900, n_features=15, noise=4.0, random_state=seed)
        y = y + 5.0 * np.sin(X[:, 0]) + 2.0 * (X[:, 1] ** 2)
        return X, y
    if name == "heteroscedastic_regression":
        X, y = make_regression(n_samples=900, n_features=25, n_informative=10,
            noise=1.0, random_state=seed)
        rng = np.random.default_rng(seed)
        y = y + rng.normal(0, 0.5 + 0.15 * np.abs(X[:, 0]) * 10, size=len(y))
        return X, y
    if name == "highdim_regression":
        return make_regression(n_samples=700, n_features=150, n_informative=12,
            noise=25.0, random_state=seed)
    if name == "smalln_regression":
        return make_regression(n_samples=220, n_features=70, n_informative=7,
            noise=20.0, random_state=seed)
    raise KeyError("Unknown data set: " + str(name))


# -------------------------- FINITE SEARCH-SPACE SPECIFICATIONS --------------------------
# Each parameter has an explicit finite candidate list. Conditional parameters
# refer to an earlier parent and are active only for listed parent values.
def space_spec(model_name):
    if model_name in ("svc", "svr"):
        return {
            "model__kernel": {"kind":"categorical", "values":["linear","rbf","poly"]},
            "model__C": {"kind":"float", "scale":"log", "values":[0.003,0.03,0.3,3.0,30.0,300.0,3000.0]},
            "model__gamma": {"kind":"float", "scale":"log", "values":[0.00001,0.0001,0.001,0.01,0.1,1.0,10.0],
                "parent":"model__kernel", "active_values":["rbf","poly"]},
            "model__degree": {"kind":"int", "values":[2,3,4,5,6],
                "parent":"model__kernel", "active_values":["poly"]},
            **({"model__epsilon":{"kind":"float","scale":"log","values":[0.001,0.01,0.1,1.0,10.0]}}
               if model_name=="svr" else {})
        }
    if model_name in ("rf", "extra", "rfr", "extr"):
        return {
            "model__n_estimators":{"kind":"int","values":[80,160,280]},
            "model__max_depth":{"kind":"categorical","values":[None,5,12,24]},
            "model__min_samples_split":{"kind":"int","values":[2,5,10]},
            "model__min_samples_leaf":{"kind":"int","values":[1,2,4]},
            "model__max_features":{"kind":"categorical","values":["sqrt",0.5,1.0]}
        }
    if model_name in ("hist", "histr"):
        return {
            "model__learning_rate":{"kind":"float","scale":"log","values":[0.025,0.06,0.12]},
            "model__max_iter":{"kind":"int","values":[80,160,260]},
            "model__max_leaf_nodes":{"kind":"int","values":[15,31,63]},
            "model__l2_regularization":{"kind":"float","values":[0.0,0.1,1.0]}
        }
    if model_name in ("knn", "knnr"):
        return {
            "model__n_neighbors":{"kind":"int","values":[3,5,9,15,25]},
            "model__weights":{"kind":"categorical","values":["uniform","distance"]},
            "model__p":{"kind":"int","values":[1,2]}
        }
    if model_name == "mlp":
        return {
            "model__solver":{"kind":"categorical","values":["adam","sgd","lbfgs"]},
            "model__hidden_layer_sizes":{"kind":"categorical","values":[(32,),(64,),(64,32)]},
            "model__alpha":{"kind":"float","scale":"log","values":[0.0001,0.001,0.01]},
            "model__learning_rate_init":{"kind":"float","scale":"log","values":[0.0005,0.003,0.015],
                "parent":"model__solver","active_values":["adam","sgd"]},
            "model__momentum":{"kind":"float","values":[0.5,0.8,0.95],
                "parent":"model__solver","active_values":["sgd"]},
            "model__beta_1":{"kind":"float","values":[0.8,0.9,0.99],
                "parent":"model__solver","active_values":["adam"]}
        }
    if model_name == "ridge":
        return {
            "model__alpha":{"kind":"float","scale":"log","values":[0.001,0.03,0.3,3.0,30.0,100.0]},
            "model__fit_intercept":{"kind":"categorical","values":[True,False]}
        }
    raise KeyError("Unknown model specification: " + str(model_name))


def make_estimator(model_name, seed):
    if model_name == "svc": est = SVC(random_state=seed, cache_size=512)
    elif model_name == "svr": est = SVR(cache_size=512)
    elif model_name == "rf": est = RandomForestClassifier(random_state=seed, n_jobs=1)
    elif model_name == "extra": est = ExtraTreesClassifier(random_state=seed, n_jobs=1)
    elif model_name == "hist": est = HistGradientBoostingClassifier(random_state=seed)
    elif model_name == "knn": est = KNeighborsClassifier()
    elif model_name == "mlp": est = MLPClassifier(random_state=seed, max_iter=350, early_stopping=True)
    elif model_name == "ridge": est = Ridge()
    elif model_name == "rfr": est = RandomForestRegressor(random_state=seed, n_jobs=1)
    elif model_name == "extr": est = ExtraTreesRegressor(random_state=seed, n_jobs=1)
    elif model_name == "histr": est = HistGradientBoostingRegressor(random_state=seed)
    elif model_name == "knnr": est = KNeighborsRegressor()
    else: raise KeyError(model_name)
    return Pipeline([("scale", StandardScaler()), ("model", est)])


# 22 heterogeneous HPO tasks: 12 classification + 10 regression. SVC/SVR include larger conditional grids; MLP adds solver-dependent parameters.
# Tasks intentionally combine different datasets, estimator families, dimensions,
# noise levels, sample sizes, imbalance, nonlinearity and conditional spaces.
TASK_CATALOG = [
    ("clf_bc_svc_conditional","breast_cancer","svc","classification"),
    ("clf_digits_rf","digits","rf","classification"),
    ("clf_digits_knn","digits","knn","classification"),
    ("clf_wine_hist","wine","hist","classification"),
    ("clf_iris_svc_conditional","iris","svc","classification"),
    ("clf_imbalanced_rf","imbalanced","rf","classification"),
    ("clf_noisy_extra","noisy_classification","extra","classification"),
    ("clf_moons_mlp","moons","mlp","classification"),
    ("clf_circles_svc_conditional","circles","svc","classification"),
    ("clf_highdim_hist","highdim_classification","hist","classification"),
    ("clf_smalln_svc","smalln_wide","svc","classification"),
    ("clf_bc_extra","breast_cancer","extra","classification"),
    ("reg_diabetes_ridge","diabetes","ridge","regression"),
    ("reg_diabetes_rf","diabetes","rfr","regression"),
    ("reg_linnerud_ridge","linnerud","ridge","regression"),
    ("reg_friedman1_extra","friedman1","extr","regression"),
    ("reg_friedman1_svr_conditional","friedman1","svr","regression"),
    ("reg_friedman2_hist","friedman2","histr","regression"),
    ("reg_friedman3_rf","friedman3","rfr","regression"),
    ("reg_sparse_ridge","sparse_regression","ridge","regression"),
    ("reg_nonlinear_knn","nonlinear_regression","knnr","regression"),
    ("reg_heteroscedastic_hist","heteroscedastic_regression","histr","regression"),
]
assert len(TASK_CATALOG) >= 20 and len({x[0] for x in TASK_CATALOG}) == len(TASK_CATALOG)

def build_task(task_id, seed=SEED_BASE):
    row = next(x for x in TASK_CATALOG if x[0] == task_id)
    name, data_name, model_name, task_type = row
    X, y = load_problem_data(data_name, seed=SEED_BASE)  # fixed benchmark data across replicates
    estimator = make_estimator(model_name, seed=seed)
    spec = space_spec(model_name)
    scoring = "f1_macro" if task_type == "classification" else "neg_root_mean_squared_error"
    score_upper_bound = 1.0 if task_type == "classification" else 0.0
    return {
        "task_id":name, "dataset":data_name, "model_family":model_name,
        "task_type":task_type, "X":X, "y":y, "estimator":estimator,
        "spec":spec, "scoring":scoring, "score_upper_bound":score_upper_bound,
        "n_samples":int(X.shape[0]), "n_features":int(X.shape[1]),
        "space_upper_bound":int(np.prod([len(v["values"]) for v in spec.values()]))
    }

print("Task count:", len(TASK_CATALOG))
pd.DataFrame(TASK_CATALOG, columns=["task_id","dataset","model","type"])

In [ ]:
# -------------------------- SPACE CONVERSION / COMMON EVALUATION --------------------------
def to_ags_space(spec):
    out = {}
    for name, item in spec.items():
        values = item["values"]
        kind = item["kind"]
        if kind == "float": base = NumericParameter(values, scale=item.get("scale","linear"))
        elif kind == "int": base = IntegerParameter(values)
        elif kind == "ordinal": base = OrdinalParameter(values)
        else: base = CategoricalParameter(values)
        if "parent" in item:
            base = ConditionalParameter(base, parent=item["parent"], active_values=item["active_values"])
        out[name] = base
    return out


def canonical_json(config):
    return json.dumps(config, sort_keys=True, default=lambda x: {"__repr__":repr(x)}, separators=(",",":"))


def active_config(params, spec):
    result = {}
    for name, item in spec.items():
        if "parent" in item:
            parent = item["parent"]
            if parent not in result or result[parent] not in item["active_values"]:
                continue
        result[name] = params[name]
    return result


def optuna_suggest_config(trial, spec):
    params = {}
    for name, item in spec.items():
        if "parent" in item:
            parent = item["parent"]
            if parent not in params or params[parent] not in item["active_values"]:
                continue
        values = list(item["values"])
        # Optuna categorical distributions support None/bool/int/float/str,
        # not tuple-valued sklearn parameters such as hidden_layer_sizes.
        safe = all(v is None or isinstance(v, (bool, int, float, str)) for v in values)
        if safe:
            params[name] = trial.suggest_categorical(name, values)
        else:
            token = trial.suggest_categorical(name, ["__AGS_CHOICE_{}__".format(i) for i in range(len(values))])
            idx = int(token.replace("__AGS_CHOICE_", "").replace("__", ""))
            params[name] = values[idx]
    return params


def random_config(spec, rng):
    params = {}
    for name, item in spec.items():
        if "parent" in item:
            parent = item["parent"]
            if parent not in params or params[parent] not in item["active_values"]:
                continue
        params[name] = item["values"][int(rng.integers(0, len(item["values"])))]
    return params


def make_split_indices(X, y, task_type, seed):
    if task_type == "classification":
        splitter = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=seed)
        return list(splitter.split(X, y))
    splitter = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=seed)
    return list(splitter.split(X, y))


def _fit_score_fold(estimator, params, X, y, train_idx, valid_idx, scoring):
    model = clone(estimator)
    model.set_params(**params)
    model.fit(X[train_idx], y[train_idx])
    return float(get_scorer(scoring)(model, X[valid_idx], y[valid_idx]))


def evaluate_config(estimator, params, X, y, folds, scoring, fold_jobs=1):
    jobs = (delayed(_fit_score_fold)(estimator, params, X, y, tr, va, scoring)
            for tr, va in folds)
    if fold_jobs == 1:
        values = [_fit_score_fold(estimator, params, X, y, tr, va, scoring) for tr, va in folds]
    else:
        values = Parallel(n_jobs=fold_jobs, prefer="processes")(jobs)
    return float(np.mean(values)), [float(v) for v in values]


def heldout_score(estimator, params, X_train, y_train, X_test, y_test, scoring):
    model = clone(estimator)
    model.set_params(**params)
    model.fit(X_train, y_train)
    return float(get_scorer(scoring)(model, X_test, y_test))


def prepare_replicate(task, seed):
    stratify = task["y"] if task["task_type"] == "classification" else None
    X_train, X_test, y_train, y_test = train_test_split(
        task["X"], task["y"], test_size=TEST_SIZE, random_state=seed,
        stratify=stratify
    )
    folds = make_split_indices(X_train, y_train, task["task_type"], seed)
    return X_train, X_test, y_train, y_test, folds


def result_key(task_id, budget, seed, method, experiment="main"):
    return "|".join(map(str, [experiment,task_id,budget,seed,method]))


def read_jsonl(path):
    rows = []
    if Path(path).exists():
        with open(path, "r", encoding="utf-8") as handle:
            for line in handle:
                line=line.strip()
                if line:
                    try: rows.append(json.loads(line))
                    except json.JSONDecodeError: warnings.warn("Skipping incomplete JSONL row")
    return rows


def append_jsonl(path, row):
    with open(path, "a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, default=lambda x: repr(x)) + "\n")
        handle.flush()
        os.fsync(handle.fileno())


def completed_keys(path):
    return {r.get("run_key") for r in read_jsonl(path) if r.get("status") == "ok"}


def base_record(task, seed, budget, method, experiment="main"):
    return {
        "run_key":result_key(task["task_id"],budget,seed,method,experiment),
        "experiment":experiment, "task_id":task["task_id"],
        "dataset":task["dataset"], "task_type":task["task_type"],
        "model_family":task["model_family"], "seed":int(seed),
        "budget":int(budget), "method":method,
        "n_samples":task["n_samples"], "n_features":task["n_features"],
        "space_upper_bound":task["space_upper_bound"],
        "scoring":task["scoring"], "status":"running",
        "timestamp_utc":datetime.now(timezone.utc).isoformat()
    }


def finish_record(record, best_params, best_cv, n_unique, fold_fits, elapsed,
                  X_train, y_train, X_test, y_test, task):
    test = heldout_score(task["estimator"],best_params,X_train,y_train,X_test,y_test,task["scoring"])
    record.update({
        "best_params":best_params, "best_cv":float(best_cv),
        "test_score":float(test), "unique_configs":int(n_unique),
        "fold_fits":int(fold_fits), "wall_seconds":float(elapsed),
        "status":"ok", "timestamp_finished_utc":datetime.now(timezone.utc).isoformat()
    })
    return record

In [ ]:
# -------------------------- OPTIMIZER RUNNERS --------------------------
def run_random_without_replacement(task, X_train, y_train, folds, budget, seed, fold_jobs=1):
    start=time.perf_counter()
    rng=np.random.default_rng(seed)
    cache={}
    attempts=0
    while len(cache)<budget and attempts < budget*100:
        attempts += 1
        params=active_config(random_config(task["spec"],rng),task["spec"])
        key=canonical_json(params)
        if key in cache: continue
        score, fold_scores=evaluate_config(task["estimator"],params,X_train,y_train,folds,task["scoring"],fold_jobs)
        cache[key]={"params":params,"score":score,"folds":len(fold_scores)}
    if not cache: raise RuntimeError("Random search found no legal configuration.")
    best=max(cache.values(),key=lambda z:z["score"])
    return best["params"],best["score"],len(cache),sum(x["folds"] for x in cache.values()),time.perf_counter()-start,0


def run_optuna_tpe(task, X_train, y_train, folds, budget, seed, variant="default", fold_jobs=1):
    start=time.perf_counter()
    dedup=(variant=="deduplicated")
    if variant=="default":
        sampler=TPESampler(seed=seed, n_startup_trials=min(10,max(5,budget//4)))
    elif variant=="multivariate":
        sampler=TPESampler(seed=seed, n_startup_trials=min(10,max(5,budget//4)),
                           multivariate=True, group=True, constant_liar=True, n_ei_candidates=48)
    elif dedup:
        sampler=TPESampler(seed=seed, n_startup_trials=min(10,max(5,budget//4)),
                           multivariate=True, group=True, constant_liar=True, n_ei_candidates=48)
    else: raise ValueError(variant)

    study=optuna.create_study(direction="maximize",sampler=sampler)
    cache={}
    duplicate_suggestions=0
    fold_fits_total=0
    objective_evaluations=0
    max_trials=budget*(RANDOM_TRIAL_MULTIPLIER if dedup else 1)
    attempts=0
    while len(cache)<budget and attempts<max_trials:
        trial=study.ask()
        params=active_config(optuna_suggest_config(trial,task["spec"]),task["spec"])
        key=canonical_json(params)
        if key in cache:
            duplicate_suggestions+=1
            if dedup:
                score=cache[key]["score"]
            else:
                score, fold_scores=evaluate_config(task["estimator"],params,X_train,y_train,folds,task["scoring"],fold_jobs)
                fold_fits_total += len(fold_scores)
                objective_evaluations += 1
                cache[key]={"params":params,"score":score,"folds":len(fold_scores)}
        else:
            score, fold_scores=evaluate_config(task["estimator"],params,X_train,y_train,folds,task["scoring"],fold_jobs)
            fold_fits_total += len(fold_scores)
            objective_evaluations += 1
            cache[key]={"params":params,"score":score,"folds":len(fold_scores)}
        study.tell(trial,score)
        attempts+=1
    if not cache: raise RuntimeError("Optuna did not complete an objective.")
    best=max(cache.values(),key=lambda z:z["score"])
    return best["params"],best["score"],len(cache),fold_fits_total,time.perf_counter()-start,duplicate_suggestions,objective_evaluations


def build_smac_space(spec, seed):
    from ConfigSpace import OrdinalHyperparameter
    cs=ConfigurationSpace(seed=seed)
    hyperparameters={}
    encoded_choices={}
    decoded_choices={}
    for name,item in spec.items():
        values=list(item["values"])
        if item["kind"] in ("float","int","ordinal"):
            # Preserve finite numeric ordering for SMAC's random-forest model.
            hp=OrdinalHyperparameter(name, sequence=values)
        else:
            # Encode non-scalar categories (None, tuples) as safe string tokens.
            tokens=["__AGS_CHOICE_{}__".format(i) for i in range(len(values))]
            hp=CategoricalHyperparameter(name, choices=tokens)
            encoded_choices[name]={v:t for v,t in zip(values,tokens)}
            decoded_choices[name]={t:v for v,t in zip(values,tokens)}
        cs.add_hyperparameter(hp)
        hyperparameters[name]=hp

    def parent_value_for_smac(parent_name, value):
        return encoded_choices.get(parent_name, {}).get(value, value)

    conditions=[]
    for name,item in spec.items():
        if "parent" not in item: continue
        child=hyperparameters[name]
        parent=hyperparameters[item["parent"]]
        active=[parent_value_for_smac(item["parent"],v) for v in item["active_values"]]
        if len(active)==1:
            conditions.append(EqualsCondition(child,parent,active[0]))
        else:
            conditions.append(InCondition(child,parent,active))
    if conditions: cs.add_conditions(conditions)
    return cs, decoded_choices


def run_smac3(task, X_train, y_train, folds, budget, seed, fold_jobs=1, walltime_limit=None):
    if not HAS_SMAC:
        raise ImportError("SMAC3/ConfigSpace unavailable: "+str(SMAC_IMPORT_ERROR))
    start=time.perf_counter()
    cs, decoded_choices=build_smac_space(task["spec"],seed)
    records={}
    fold_fits_total=0
    objective_evaluations=0
    def objective(config, seed=seed):
        nonlocal fold_fits_total, objective_evaluations
        raw=dict(config)
        decoded={name:decoded_choices.get(name,{}).get(value,value) for name,value in raw.items()}
        params=active_config(decoded,task["spec"])
        key=canonical_json(params)
        score, fold_scores=evaluate_config(task["estimator"],params,X_train,y_train,folds,task["scoring"],fold_jobs)
        records[key]={"params":params,"score":score,"folds":len(fold_scores)}
        fold_fits_total += len(fold_scores)
        objective_evaluations += 1
        return -float(score)  # SMAC minimizes cost
    scenario_kwargs={"deterministic":True,"n_trials":int(budget)}
    if walltime_limit is not None:
        scenario_kwargs["walltime_limit"]=float(walltime_limit)
    scenario=Scenario(cs,**scenario_kwargs)
    facade=HyperparameterOptimizationFacade(scenario,objective,overwrite=True,seed=seed)
    facade.optimize()
    if not records: raise RuntimeError("SMAC returned no completed configurations.")
    best=max(records.values(),key=lambda z:z["score"])
    return best["params"],best["score"],len(records),fold_fits_total,time.perf_counter()-start,0,objective_evaluations


def run_ags(task, X_train, y_train, folds, budget, seed, n_jobs=1,
            extra_kwargs=None, time_budget=None, disable_respawn=False):
    start=time.perf_counter()
    kwargs=dict(
        estimator=task["estimator"], param_grid=to_ags_space(task["spec"]),
        cv=CV_FOLDS, scoring=task["scoring"], max_evaluations=int(budget),
        n_jobs=n_jobs, random_state=seed, profiling=True
    )
    kwargs.update(extra_kwargs or {})
    obj=AdaptiveGreedySearch(**kwargs)
    if disable_respawn:
        obj._respawn=MethodType(lambda self, retired: None,obj)
    if time_budget is not None:
        deadline=time.perf_counter()+float(time_budget)
        original_propose=obj._propose
        def deadline_guard(self):
            if time.perf_counter()>=deadline:
                for climber in self.climbers:
                    if climber.status=="active": climber.status="stagnated"
                return {},[]
            return original_propose()
        obj._propose=MethodType(deadline_guard,obj)
        # fit() then exits cleanly at the next batch boundary. An in-progress
        # fold/batch is not forcibly killed; actual elapsed time is recorded.
        kwargs["max_evaluations"]=max(int(budget),obj.search_space_size_upper_bound)
        obj.max_evaluations=kwargs["max_evaluations"]
    obj.fit(X_train,y_train)
    # Enforce identical inner CV splits across AGS and every other baseline.
    if len(obj._cv_splits) != len(folds):
        raise AssertionError("AGS and baseline CV split counts differ.")
    for (ags_train,ags_valid),(ref_train,ref_valid) in zip(obj._cv_splits,folds):
        if not (np.array_equal(ags_train,ref_train) and np.array_equal(ags_valid,ref_valid)):
            raise AssertionError("AGS and baseline CV fold indices differ.")
    elapsed=time.perf_counter()-start
    hist=list(getattr(obj,"history",[]))
    fold_fits=sum(int(h.get("n_folds_used",CV_FOLDS)) for h in hist)
    return obj.best_params,float(obj.best_score),len(obj.visited_states),fold_fits,elapsed,obj


def run_method(method, task, seed, budget, fold_jobs=1, n_jobs=1):
    X_train,X_test,y_train,y_test,folds=prepare_replicate(task,seed)
    if method=="AGS":
        params,score,n_unique,fold_fits,elapsed,extra=run_ags(task,X_train,y_train,folds,budget,seed,n_jobs=n_jobs)
        duplicate=0
    elif method=="SMAC3":
        params,score,n_unique,fold_fits,elapsed,duplicate,objective_evaluations=run_smac3(task,X_train,y_train,folds,budget,seed,fold_jobs)
        extra=None
    elif method=="Optuna_TPE_default":
        params,score,n_unique,fold_fits,elapsed,duplicate,objective_evaluations=run_optuna_tpe(task,X_train,y_train,folds,budget,seed,"default",fold_jobs)
        extra=None
    elif method=="Optuna_TPE_multivariate":
        params,score,n_unique,fold_fits,elapsed,duplicate,objective_evaluations=run_optuna_tpe(task,X_train,y_train,folds,budget,seed,"multivariate",fold_jobs)
        extra=None
    elif method=="Optuna_TPE_deduplicated":
        params,score,n_unique,fold_fits,elapsed,duplicate,objective_evaluations=run_optuna_tpe(task,X_train,y_train,folds,budget,seed,"deduplicated",fold_jobs)
        extra=None
    elif method=="Random_without_replacement":
        params,score,n_unique,fold_fits,elapsed,duplicate=run_random_without_replacement(task,X_train,y_train,folds,budget,seed,fold_jobs)
        objective_evaluations=n_unique
        extra=None
    else: raise KeyError(method)
    rec=base_record(task,seed,budget,method)
    rec=finish_record(rec,params,score,n_unique,fold_fits,elapsed,X_train,y_train,X_test,y_test,task)
    rec["duplicate_suggestions"]=int(duplicate)
    rec["objective_evaluations"]=int(objective_evaluations if method!="AGS" else n_unique)
    if method=="AGS":
        rec.update({
            "pruned":int(getattr(extra,"n_pruned",0)),
            "folds_saved":int(getattr(extra,"folds_saved",0)),
            "mean_fidelity":float(getattr(extra,"mean_fidelity",1.0)),
            "climbers_spawned":int(getattr(extra,"climbers_spawned",0)),
            "climbers_merged":int(getattr(extra,"climbers_merged",0)),
            "trust_expansions":int(getattr(extra,"trust_expansions",0)),
            "trust_contractions":int(getattr(extra,"trust_contractions",0)),
            "profile_report":extra.profile_report(print_report=False)
        })
    return rec

## 22-task benchmark inventory

The benchmark includes classification and regression; clean and noisy tasks; low- and high-dimensional inputs; small-`n)/wide-`p`; imbalance; nonlinear decision boundaries; and conditional SVM/SVR spaces.

**Why not directly drop YAHPO Gym into this notebook?** YAHPO evaluates a supplied configuration against a pre-trained surrogate benchmark and returns objective values (often with resource/fidelity parameters). AGS 3.7.1 accepts a scikit-learn estimator and internally generates CV folds. A naïve wrapper would repeat the same YAHPO objective across folds and create pseudo-replication. The optional section records how to inventory YAHPO scenarios, but a fair AGS-vs-SMAC YAHPO experiment requires adding an objective-function + fidelity interface to AGS first. See the [YAHPO Getting Started guide](https://slds-lmu.github.io/yahpo_gym/getting_started.html) and [scenario overview](https://slds-lmu.github.io/yahpo_gym/scenarios.html).

In [ ]:
# Confirm all tasks load, all parameter spaces are finite, and the conditional SVM is legal.
inventory=[]
for task_id,_,_,_ in TASK_CATALOG:
    t=build_task(task_id)
    inventory.append({k:t[k] for k in ["task_id","dataset","model_family","task_type","n_samples","n_features","space_upper_bound","scoring"]})
TASKS_DF=pd.DataFrame(inventory)
display(TASKS_DF)
assert len(TASKS_DF)>=20
assert TASKS_DF.task_id.nunique()==len(TASKS_DF)

# Pilot: one task, tiny budgets, one seed. Tests the complete baseline pipeline.
PILOT_TASK="clf_bc_svc_conditional"
pilot=build_task(PILOT_TASK)
pilot_rows=[]
for method in ["AGS","Optuna_TPE_default","Optuna_TPE_multivariate","Optuna_TPE_deduplicated","Random_without_replacement"] + (["SMAC3"] if HAS_SMAC else []):
    try:
        row=run_method(method,pilot,seed=SEED_BASE,budget=8,fold_jobs=1,n_jobs=1)
        pilot_rows.append({k:row.get(k) for k in ["method","status","best_cv","test_score","unique_configs","fold_fits","wall_seconds","duplicate_suggestions"]})
    except Exception as exc:
        pilot_rows.append({"method":method,"status":"error","error":repr(exc)})
display(pd.DataFrame(pilot_rows))
if any(r.get("status")=="error" for r in pilot_rows):
    raise RuntimeError("Resolve pilot errors before launching the full benchmark.")

## Main benchmark: equal objective-trial / visit budgets

The primary comparison is AGS vs SMAC3, three Optuna configurations and random sampling without replacement. Each method receives the same seed-specific outer split, inner folds, search-space definition and objective-trial/visit budget. The notebook saves one complete result row at a time and supports resume after interruption.

**Important fairness note:** AGS fold-level pruning changes the number of fold-fits, while other methods perform full CV. We therefore report both unique configurations and fold-fits. The primary equal-budget comparison is not a claim of equal compute; the fold-fit and equal-time experiments below address compute efficiency separately.

In [ ]:
METHODS_MAIN=[
    "AGS","SMAC3","Optuna_TPE_default","Optuna_TPE_multivariate",
    "Optuna_TPE_deduplicated","Random_without_replacement"
]
if not HAS_SMAC:
    print("SMAC3 will be recorded as unavailable; install smac and ConfigSpace, restart, then rerun.")
    
def run_main_benchmark(task_ids=None, seeds=None, budgets=None):
    task_ids=task_ids or [x[0] for x in TASK_CATALOG]
    seeds=seeds if seeds is not None else list(range(SEED_BASE,SEED_BASE+N_SEEDS))
    budgets=budgets or BUDGETS
    done=completed_keys(RAW_PATH) if RESUME else set()
    rows=read_jsonl(RAW_PATH) if RESUME else []
    total=len(task_ids)*len(seeds)*len(budgets)*len(METHODS_MAIN)
    count=0
    for task_id in task_ids:
        task=build_task(task_id)
        for budget in budgets:
            for seed in seeds:
                for method in METHODS_MAIN:
                    key=result_key(task_id,budget,seed,method)
                    if key in done:
                        count+=1; continue
                    rec=base_record(task,seed,budget,method)
                    if method=="SMAC3" and not HAS_SMAC:
                        rec.update(status="skipped",error=str(SMAC_IMPORT_ERROR))
                        append_jsonl(RAW_PATH,rec); rows.append(rec); count+=1; continue
                    try:
                        rec=run_method(method,task,seed,budget,fold_jobs=1,n_jobs=1)
                    except Exception as exc:
                        rec.update(status="error",error=repr(exc),traceback=__import__("traceback").format_exc())
                    append_jsonl(RAW_PATH,rec); rows.append(rec); count+=1
                    if count%10==0:
                        print(f"{count}/{total}: {task_id} | {budget} | seed={seed} | {method} | {rec.get('status')}")
    return pd.DataFrame(rows)

MAIN_RESULTS=run_main_benchmark()
display(MAIN_RESULTS.groupby(["task_id","budget","method","status"],dropna=False).size().rename("runs").reset_index().head(50))
print("Raw records:",len(MAIN_RESULTS),"->",RAW_PATH.resolve())

## Component ablations

The ablation suite isolates one design choice at a time. It runs on six deliberately varied representative tasks rather than all 22 to keep the factorial cost tractable. Increase `ABLATION_TASKS`, seeds and budget for the final manuscript.

- **Swarm size:** 1 vs default vs 8 climbers.
- **Trust-region adaptation:** disable adaptation by setting expansion/shrinkage multipliers to 1.
- **Diversity:** zero local and restart diversity weights.
- **Respawning:** disable `_respawn` while leaving the rest unchanged.
- **Warm-up:** minimum observations 2 vs the default vs 20.
- **Pruning:** none, percentile, racing, and optimistic with an explicit mathematically valid scorer upper bound.

Ablation rows include the configuration overrides. Do not interpret an ablation as causal if multiple settings differ at once.

In [ ]:
ABLATION_TASKS=[
    "clf_bc_svc_conditional","clf_highdim_hist","clf_imbalanced_rf",
    "reg_diabetes_ridge","reg_friedman1_svr_conditional","reg_heteroscedastic_hist"
]
ABLATION_VARIANTS={
    "default": {},
    "swarm_1": {"n_climbers":1},
    "swarm_8": {"n_climbers":8},
    "no_trust_adaptation": {"trust_expand":1.0,"trust_shrink":1.0},
    "no_diversity": {"diversity_weight":0.0,"restart_diversity_weight":0.0},
    "no_respawn": {"__disable_respawn__":True},
    "warmup_2": {"min_observations":2},
    "warmup_20": {"min_observations":20},
    "no_pruning": {"pruning_strategy":"none"},
    "percentile_pruning": {"pruning_strategy":"percentile"},
    "racing_pruning": {"pruning_strategy":"racing"},
    "optimistic_pruning": {"pruning_strategy":"optimistic"}
}

def run_ablation(task_id, seed, budget, variant, overrides):
    task=build_task(task_id)
    X_train,X_test,y_train,y_test,folds=prepare_replicate(task,seed)
    extra={k:v for k,v in overrides.items() if not k.startswith("__")}
    if extra.get("pruning_strategy")=="optimistic":
        extra["score_upper_bound"]=task["score_upper_bound"]
    params,score,n_unique,fold_fits,elapsed,obj=run_ags(
        task,X_train,y_train,folds,budget,seed,n_jobs=1,
        extra_kwargs=extra,disable_respawn=bool(overrides.get("__disable_respawn__",False))
    )
    rec=base_record(task,seed,budget,"AGS_"+variant,"ablation")
    rec=finish_record(rec,params,score,n_unique,fold_fits,elapsed,X_train,y_train,X_test,y_test,task)
    rec.update({"variant":variant,"overrides":extra,
        "pruned":int(obj.n_pruned),"folds_saved":int(obj.folds_saved),
        "climbers_spawned":int(obj.climbers_spawned),
        "trust_expansions":int(obj.trust_expansions),
        "trust_contractions":int(obj.trust_contractions),
        "mean_fidelity":float(obj.mean_fidelity)})
    return rec

def run_ablation_suite():
    done=completed_keys(ABLATION_PATH) if RESUME else set()
    rows=read_jsonl(ABLATION_PATH) if RESUME else []
    seeds=list(range(SEED_BASE,SEED_BASE+(1 if QUICK_MODE else min(N_SEEDS,10))))
    for task_id in ABLATION_TASKS:
        for seed in seeds:
            for variant,overrides in ABLATION_VARIANTS.items():
                budget=ABLATION_BUDGET
                key=result_key(task_id,budget,seed,"AGS_"+variant,"ablation")
                if key in done: continue
                try: rec=run_ablation(task_id,seed,budget,variant,overrides)
                except Exception as exc:
                    task=build_task(task_id); rec=base_record(task,seed,budget,"AGS_"+variant,"ablation")
                    rec.update(status="error",variant=variant,error=repr(exc),traceback=__import__("traceback").format_exc())
                append_jsonl(ABLATION_PATH,rec); rows.append(rec)
        print("Ablations:",task_id,"complete")
    return pd.DataFrame(rows)

ABLATION_RESULTS=run_ablation_suite()
display(ABLATION_RESULTS.groupby(["variant","status"],dropna=False).agg(
    runs=("run_key","count"),mean_test=("test_score","mean"),
    mean_folds=("fold_fits","mean"),mean_wall=("wall_seconds","mean")).reset_index())

## Pruning safety and false-pruning audit

For each representative task, AGS is run with `none`, `percentile`, `racing` and `optimistic` pruning. Every configuration pruned by a heuristic is then **re-evaluated to completion** on the exact same CV folds. This counterfactual audit estimates how often a discarded configuration would have beaten the best completed configuration found by that run.

This is a conservative, transparent diagnostic—not an exact family-wise false-pruning probability. In parallel batches, AGS workers share the incumbent snapshot from batch dispatch; the API history does not currently store that snapshot per candidate. The notebook therefore reports a clear proxy: fraction of pruned configurations whose completed score reaches or exceeds the final incumbent (and their regret relative to it). For optimistic pruning, the configured score bound must be valid: F1/macro-F1 is bounded above by 1; negative RMSE is bounded above by 0.

In [ ]:
def run_pruning_audit(task_id,seed,budget,strategy):
    task=build_task(task_id)
    X_train,X_test,y_train,y_test,folds=prepare_replicate(task,seed)
    kwargs={"pruning_strategy":strategy}
    if strategy=="optimistic": kwargs["score_upper_bound"]=task["score_upper_bound"]
    params,best_cv,n_unique,fold_fits,elapsed,obj=run_ags(
        task,X_train,y_train,folds,budget,seed,n_jobs=1,extra_kwargs=kwargs
    )
    pruned=[h for h in obj.history if h.get("pruned")]
    counterfactual=[]
    for h in pruned:
        full_score,fold_scores=evaluate_config(
            task["estimator"],h["params"],X_train,y_train,folds,task["scoring"],fold_jobs=1
        )
        counterfactual.append({
            "prune_reason":h.get("prune_reason"),
            "partial_mean":float(h.get("partial_mean",np.nan)),
            "folds_used":int(h.get("n_folds_used",0)),
            "full_cv_counterfactual":float(full_score),
            "would_match_or_beat_final_incumbent":bool(full_score>=best_cv-1e-12),
            "regret_vs_final_incumbent":float(max(0.0,best_cv-full_score))
        })
    false_count=sum(x["would_match_or_beat_final_incumbent"] for x in counterfactual)
    rec={
        "run_key":result_key(task_id,budget,seed,"AGS_"+strategy,"pruning_audit"),
        "experiment":"pruning_audit","task_id":task_id,"seed":seed,"budget":budget,
        "strategy":strategy,"best_cv":float(best_cv),
        "n_pruned":len(pruned),"counterfactual_fits":len(counterfactual)*CV_FOLDS,
        "proxy_false_prunes":false_count,
        "proxy_false_prune_rate":(false_count/len(pruned) if pruned else np.nan),
        "mean_pruned_regret":(float(np.mean([x["regret_vs_final_incumbent"] for x in counterfactual])) if counterfactual else np.nan),
        "counterfactuals":counterfactual,"wall_seconds":float(elapsed),
        "status":"ok"
    }
    return rec

def run_pruning_suite():
    done=completed_keys(PRUNING_PATH) if RESUME else set()
    rows=read_jsonl(PRUNING_PATH) if RESUME else []
    strategies=["none","percentile","racing","optimistic"]
    seeds=list(range(SEED_BASE,SEED_BASE+(1 if QUICK_MODE else min(N_SEEDS,10))))
    for task_id in ABLATION_TASKS:
        for seed in seeds:
            for strategy in strategies:
                budget=PRUNING_BUDGET
                key=result_key(task_id,budget,seed,"AGS_"+strategy,"pruning_audit")
                if key in done: continue
                try: rec=run_pruning_audit(task_id,seed,budget,strategy)
                except Exception as exc:
                    rec={"run_key":key,"experiment":"pruning_audit","task_id":task_id,
                         "seed":seed,"budget":budget,"strategy":strategy,
                         "status":"error","error":repr(exc),
                         "traceback":__import__("traceback").format_exc()}
                append_jsonl(PRUNING_PATH,rec); rows.append(rec)
        print("Pruning audit:",task_id,"complete")
    return pd.DataFrame(rows)

PRUNING_RESULTS=run_pruning_suite()
display(PRUNING_RESULTS.groupby(["strategy","status"],dropna=False).agg(
    runs=("run_key","count"),pruned=("n_pruned","mean"),
    proxy_false_prune_rate=("proxy_false_prune_rate","mean"),
    mean_regret=("mean_pruned_regret","mean")).reset_index())

## Equal-wall-clock and multicore comparisons

AGS does not expose a native wall-time stopping parameter in v3.7.1. The AGS runner below stops **between proposal/evaluation batches** by retiring active climbers when the deadline has elapsed; an already-running batch is allowed to finish. Thus the time cap is a practical soft limit, and actual elapsed time is reported. This avoids claiming an exact hard timeout that the package does not provide.

For SMAC3, the Scenario supports a wall-clock limit. Optuna and random search run complete trials until the deadline, so the last trial may overrun. Report actual time and achieved unique configurations. Run serial and multicore conditions separately; the baseline methods use parallel CV folds, while AGS parallelizes candidate evaluations. This compares practical execution strategies under the same worker limit, not identical internal scheduling.

In [ ]:
def run_time_budgeted(method,task_id,seed,time_budget,n_jobs):
    task=build_task(task_id)
    X_train,X_test,y_train,y_test,folds=prepare_replicate(task,seed)
    start=time.perf_counter()
    if method=="AGS":
        params,score,n_unique,fold_fits,elapsed,obj=run_ags(
            task,X_train,y_train,folds,budget=task["space_upper_bound"],
            seed=seed,n_jobs=n_jobs,time_budget=time_budget
        )
        duplicate=0
    elif method=="SMAC3":
        params,score,n_unique,fold_fits,elapsed,duplicate,objective_evaluations=run_smac3(
            task,X_train,y_train,folds,budget=10**7,seed=seed,
            fold_jobs=n_jobs,walltime_limit=time_budget
        )
    else:
        cache={}
        rng=np.random.default_rng(seed)
        sampler=TPESampler(seed=seed,multivariate=True,group=True,constant_liar=True)
        study=optuna.create_study(direction="maximize",sampler=sampler)
        duplicate=0; attempts=0
        while time.perf_counter()-start<time_budget:
            trial=study.ask()
            params=active_config(optuna_suggest_config(trial,task["spec"]),task["spec"])
            key=canonical_json(params)
            if key in cache:
                duplicate+=1; score=cache[key]["score"]
            else:
                score,fs=evaluate_config(task["estimator"],params,X_train,y_train,folds,task["scoring"],fold_jobs=n_jobs)
                cache[key]={"params":params,"score":score,"folds":len(fs)}
            study.tell(trial,score); attempts+=1
        if not cache: raise RuntimeError("No trial finished before the wall-clock deadline.")
        best=max(cache.values(),key=lambda x:x["score"])
        params,score=best["params"],best["score"]
        n_unique=len(cache); fold_fits=sum(x["folds"] for x in cache.values())
        elapsed=time.perf_counter()-start
        obj=None
    rec={
        "run_key":result_key(task_id,int(time_budget),seed,method+"_jobs"+str(n_jobs),"time_parallel"),
        "experiment":"time_parallel","task_id":task_id,"seed":seed,
        "time_budget_seconds":time_budget,"method":method,"n_jobs":n_jobs,
        "best_cv":float(score),"unique_configs":int(n_unique),
        "fold_fits":int(fold_fits),"wall_seconds":float(elapsed),
        "duplicate_suggestions":int(duplicate),
        "test_score":heldout_score(task["estimator"],params,X_train,y_train,X_test,y_test,task["scoring"]),
        "best_params":params,"status":"ok"
    }
    if method=="AGS":
        rec.update({"n_pruned":int(obj.n_pruned),"folds_saved":int(obj.folds_saved),
                    "profile_report":obj.profile_report(print_report=False)})
    return rec


def run_parallel_time_suite():
    done=completed_keys(TIMING_PATH) if RESUME else set()
    rows=read_jsonl(TIMING_PATH) if RESUME else []
    ids=["clf_bc_svc_conditional","clf_highdim_hist","reg_diabetes_ridge","reg_friedman1_svr_conditional"]
    methods=["AGS","SMAC3","Optuna_TPE_multivariate"]
    jobs=[1,-1]
    seeds=[SEED_BASE] if QUICK_MODE else [SEED_BASE,SEED_BASE+1,SEED_BASE+2]
    for task_id in ids:
        for seed in seeds:
            for nj in jobs:
                for method in methods:
                    key=result_key(task_id,int(TIME_BUDGET_SECONDS),seed,method+"_jobs"+str(nj),"time_parallel")
                    if key in done: continue
                    if method=="SMAC3" and not HAS_SMAC:
                        rec={"run_key":key,"experiment":"time_parallel","status":"skipped","error":str(SMAC_IMPORT_ERROR)}
                    else:
                        try: rec=run_time_budgeted(method,task_id,seed,TIME_BUDGET_SECONDS,nj)
                        except Exception as exc:
                            rec={"run_key":key,"experiment":"time_parallel","task_id":task_id,
                                 "seed":seed,"method":method,"n_jobs":nj,"status":"error",
                                 "error":repr(exc),"traceback":__import__("traceback").format_exc()}
                    append_jsonl(TIMING_PATH,rec); rows.append(rec)
        print("Time/parallel suite:",task_id,"complete")
    return pd.DataFrame(rows)

TIMING_RESULTS=run_parallel_time_suite()
display(TIMING_RESULTS.groupby(["method","n_jobs","status"],dropna=False).agg(
    runs=("run_key","count"),mean_score=("test_score","mean"),
    mean_configs=("unique_configs","mean"),mean_wall=("wall_seconds","mean"),
    mean_fold_fits=("fold_fits","mean")).reset_index())

## Task-level statistical analysis

Do not pool raw classification F1 and negative RMSE into one mean. The analysis below:

1. Reports per-task/per-budget mean held-out score and 95% t-interval over paired seeds.
2. Computes task-level ranks within each task/seed, then summarizes mean ranks across tasks.
3. Uses Friedman tests for the omnibus comparison where at least three complete methods are available.
4. Uses paired Wilcoxon tests over dataset-clustered mean-rank differences against AGS, with Holm correction.
5. Reports paired standardized effect size (Cohen's (d_z)) and rank-biserial correlation where defined.
6. Bootstraps the problem-level average rank to obtain confidence intervals. Underlying datasets—not individual tasks, replicates or folds—are the resampling unit for cross-problem uncertainty.

The test set is the final held-out split for each seed and is only scored after the configuration has been selected.

In [ ]:
def mean_ci(values, confidence=0.95):
    v=np.asarray([x for x in values if pd.notna(x)],dtype=float)
    if len(v)==0: return np.nan,np.nan,np.nan
    m=float(np.mean(v))
    if len(v)==1: return m,np.nan,np.nan
    se=float(np.std(v,ddof=1)/np.sqrt(len(v)))
    q=float(student_t.ppf((1+confidence)/2,len(v)-1)) if HAS_SCIPY else 1.96
    return m,m-q*se,m+q*se


def holm_adjust(pvalues):
    p=np.asarray(pvalues,dtype=float)
    m=len(p); order=np.argsort(p); adjusted=np.empty(m); running=0.0
    for rank,idx in enumerate(order):
        val=min(1.0,(m-rank)*p[idx])
        running=max(running,val); adjusted[idx]=running
    return adjusted.tolist()


def paired_effect(a,b):
    # a and b must be comparable within their matched block. Cross-task inference
    # below uses task-level ranks, not raw mixed classification/regression metrics.
    a=np.asarray(a,dtype=float); b=np.asarray(b,dtype=float)
    d=a-b; d=d[np.isfinite(d)]
    if len(d)==0: return {"mean_difference":np.nan,"cohens_dz":np.nan,"rank_biserial":np.nan}
    sd=np.std(d,ddof=1) if len(d)>1 else np.nan
    dz=float(np.mean(d)/sd) if sd and np.isfinite(sd) else np.nan
    nonzero=d!=0
    if np.any(nonzero):
        abs_ranks=rankdata(np.abs(d[nonzero])) if HAS_SCIPY else np.arange(1,np.sum(nonzero)+1,dtype=float)
        signs=d[nonzero]
        w_plus=float(np.sum(abs_ranks[signs>0]))
        w_minus=float(np.sum(abs_ranks[signs<0]))
        rb=float((w_plus-w_minus)/(w_plus+w_minus)) if (w_plus+w_minus)>0 else 0.0
    else:
        rb=0.0
    return {"mean_difference":float(np.mean(d)),"cohens_dz":dz,"rank_biserial":rb}


def analyze_main_results(path=RAW_PATH):
    df=pd.DataFrame(read_jsonl(path))
    df=df[(df.status=="ok") & (df.experiment=="main")].copy()
    if df.empty: raise RuntimeError("No successful main benchmark rows found.")
    summaries=[]
    keys=["task_id","task_type","budget","method"]
    for group_key,g in df.groupby(keys):
        m,lo,hi=mean_ci(g.test_score)
        summaries.append(dict(zip(keys,group_key))|{
            "n":len(g),"mean_test_score":m,"ci95_low":lo,"ci95_high":hi,
            "mean_best_cv":g.best_cv.mean(),"mean_unique_configs":g.unique_configs.mean(),
            "mean_fold_fits":g.fold_fits.mean(),"mean_wall_seconds":g.wall_seconds.mean()
        })
    summary=pd.DataFrame(summaries)
    # Within task, budget and seed rank; no raw metric pooling across tasks.
    df["rank"]=df.groupby(["task_id","budget","seed"]).test_score.rank(ascending=False,method="average")
    rank_task=df.groupby(["task_id","budget","method"]).rank.mean().reset_index()
    rank_summary=rank_task.groupby(["budget","method"]).rank.agg(["mean","std","count"]).reset_index()
    # Aggregate tasks sharing the same source dataset to avoid treating them as
    # independent benchmark units. This also avoids mixing raw metric scales.
    task_to_dataset={x[0]:x[1] for x in TASK_CATALOG}
    rank_task["dataset"]=rank_task["task_id"].map(task_to_dataset)
    rank_dataset=rank_task.groupby(["dataset","budget","method"]).rank.mean().reset_index()

    # Paired cross-dataset inference uses dataset-level mean ranks.
    tests=[]
    rank_pivot=rank_dataset.pivot_table(index=["dataset","budget"],columns="method",values="rank")
    for budget in sorted(df.budget.unique()):
        pivot=rank_pivot.xs(budget,level="budget",drop_level=True)
        if "AGS" not in pivot.columns: continue
        methods=[m for m in pivot.columns if m!="AGS"]
        pvals=[]; pending=[]
        for method in methods:
            pair=pivot[["AGS",method]].dropna()
            # Positive rank advantage means AGS has a lower (better) rank.
            eff=paired_effect(pair[method],pair["AGS"])
            if HAS_SCIPY and len(pair)>=5:
                try: stat,p=wilcoxon(pair[method],pair["AGS"],alternative="two-sided",zero_method="wilcox")
                except Exception: stat,p=np.nan,1.0
            else: stat,p=np.nan,np.nan
            pending.append({"budget":budget,"comparison":"AGS vs "+method,
                "n_dataset_clusters":len(pair),"mean_rank_advantage":eff["mean_difference"],
                "wilcoxon_stat":stat,"p_raw":p,
                "cohens_dz_on_rank_difference":eff["cohens_dz"],
                "rank_biserial":eff["rank_biserial"]})
            pvals.append(p)
        valid=[i for i,p in enumerate(pvals) if np.isfinite(p)]
        adj=holm_adjust([pvals[i] for i in valid]) if valid else []
        for i,a in zip(valid,adj): pending[i]["p_holm"]=a
        for row in pending:
            if "p_holm" not in row: row["p_holm"]=np.nan
            tests.append(row)
    # Friedman omnibus over dataset-clustered average within-task ranks.
    friedman=[]
    for budget in sorted(df.budget.unique()):
        pivot=rank_dataset[rank_dataset.budget==budget].pivot(index="dataset",columns="method",values="rank")
        cols=[c for c in pivot.columns if pivot[c].notna().all()]
        if HAS_SCIPY and len(cols)>=3 and len(pivot)>=5:
            stat,p=friedmanchisquare(*[pivot[c].values for c in cols])
        else: stat,p=np.nan,np.nan
        friedman.append({"budget":budget,"methods":cols,"n_dataset_clusters":len(pivot),
                         "friedman_stat":stat,"p_value":p})
    # Bootstrap source-dataset clusters (all tasks from a sampled dataset stay together).
    rng=np.random.default_rng(SEED_BASE)
    boot=[]
    for (budget,method),g in rank_dataset.groupby(["budget","method"]):
        clusters={name:grp["rank"].to_numpy(dtype=float)
                  for name,grp in g.groupby("dataset")}
        names=list(clusters)
        if not names: continue
        reps=[]
        for _ in range(5000):
            sampled=rng.choice(names,size=len(names),replace=True)
            sampled_values=np.concatenate([clusters[name] for name in sampled])
            reps.append(float(np.mean(sampled_values)))
        boot.append({"budget":budget,"method":method,"mean_rank":float(g["rank"].mean()),
                     "rank_ci95_low":float(np.quantile(reps,.025)),
                     "rank_ci95_high":float(np.quantile(reps,.975)),
                     "n_dataset_clusters":len(names)})
    return summary,rank_summary,pd.DataFrame(tests),pd.DataFrame(friedman),pd.DataFrame(boot),df

SUMMARY_BY_TASK,RANK_SUMMARY,PAIRED_TESTS,FRIEDMAN_TESTS,RANK_BOOTSTRAP,ANALYSIS_RAW=analyze_main_results()
display(SUMMARY_BY_TASK.head(30))
display(RANK_SUMMARY)
display(PAIRED_TESTS)
display(FRIEDMAN_TESTS)
display(RANK_BOOTSTRAP)

SUMMARY_BY_TASK.to_csv(ARTIFACT_DIR/"summary_by_task.csv",index=False)
RANK_SUMMARY.to_csv(ARTIFACT_DIR/"rank_summary.csv",index=False)
PAIRED_TESTS.to_csv(ARTIFACT_DIR/"paired_rank_effects_holm.csv",index=False)
FRIEDMAN_TESTS.to_json(ARTIFACT_DIR/"friedman_tests.json",orient="records",indent=2)
RANK_BOOTSTRAP.to_csv(ARTIFACT_DIR/"rank_bootstrap_ci.csv",index=False)

## Optional YAHPO Gym scenario inventory (not part of the primary estimator-CV results)

YAHPO Gym provides a large suite of tabular black-box HPO benchmarks with scenario- and instance-specific configuration spaces and, in some scenarios, resource fidelity. It can substantially strengthen external validity, but the evaluator interface is different from AGS 3.7.1's `fit(X, y)` estimator interface.

The cell below is intentionally non-destructive. It checks whether YAHPO is installed, prints package/scenario metadata where supported, and saves a note. It does **not** generate fake fold replications or add YAHPO results to the main statistical table. Before using YAHPO as a headline benchmark, add a public callable-objective API to AGS (objective(config, resource) → scalar + metadata) and compare all optimizers on precisely the same YAHPO task instances, fidelity schedules and budgets.

References:
- https://slds-lmu.github.io/yahpo_gym/getting_started.html
- https://slds-lmu.github.io/yahpo_gym/scenarios.html
- https://proceedings.mlr.press/ (PMLR venue; check the specific YAHPO publication metadata before citing)

In [ ]:
YAHPO_NOTE={
    "status":"not_integrated_into_primary_results",
    "reason":"YAHPO is a surrogate black-box objective benchmark; AGS 3.7.1 is estimator/CV-based. Repeating the same black-box output across CV folds would be invalid pseudo-replication.",
    "getting_started":"https://slds-lmu.github.io/yahpo_gym/getting_started.html",
    "scenario_overview":"https://slds-lmu.github.io/yahpo_gym/scenarios.html",
    "required_ags_extension":"A public objective-function and resource/fidelity interface before a fair YAHPO comparison."
}
try:
    import yahpo_gym
    YAHPO_NOTE["installed"]=True
    YAHPO_NOTE["version"]=getattr(yahpo_gym,"__version__","unknown")
    try:
        from yahpo_gym import benchmark_set
        YAHPO_NOTE["benchmark_set_api_available"]=hasattr(benchmark_set,"BenchmarkSet")
    except Exception as exc:
        YAHPO_NOTE["api_probe_error"]=repr(exc)
except Exception as exc:
    YAHPO_NOTE["installed"]=False
    YAHPO_NOTE["import_error"]=repr(exc)

with open(ARTIFACT_DIR/"yahpo_integration_note.json","w",encoding="utf-8") as f:
    json.dump(YAHPO_NOTE,f,indent=2)
YAHPO_NOTE

## Export a complete reproducibility manifest

This cell records versions, machine metadata, protocol settings, task inventory, data-generation policy and output checksums. Keep this file with raw results. A publication should report exact seeds, package versions, CPU model/count, OS, Python version, and whether parallel runs used all cores or a fixed allocation.

Do not delete error records: failures and skipped methods should be disclosed and fixed before drawing conclusions.

In [ ]:
import importlib.metadata as ilm
def package_version(name):
    try: return ilm.version(name)
    except Exception: return None

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""): h.update(block)
    return h.hexdigest()

manifest={
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "repository":"https://github.com/MJwd-Hasan/adaptive-greedy-search",
    "repository_branch_expected":"benchmark/comprehensive-protocol-v2",
    "ags_version":getattr(__import__("ags"),"__version__","unknown"),
    "packages":{p:package_version(p) for p in [
        "adaptive-greedy-search","scikit-learn","numpy","scipy","optuna",
        "smac","ConfigSpace","joblib","pandas"
    ]},
    "python":sys.version,
    "platform":platform.platform(),
    "processor":platform.processor(),
    "cpu_count":os.cpu_count(),
    "settings":{
        "quick_mode":QUICK_MODE,"n_seeds":N_SEEDS,"cv_folds":CV_FOLDS,
        "budgets":BUDGETS,"test_size":TEST_SIZE,"n_jobs":N_JOBS,
        "ablation_budget":ABLATION_BUDGET,"pruning_budget":PRUNING_BUDGET,
        "time_budget_seconds":TIME_BUDGET_SECONDS
    },
    "task_catalog":[list(x) for x in TASK_CATALOG],
    "primary_methods":METHODS_MAIN,
    "random_seeds":list(range(SEED_BASE,SEED_BASE+N_SEEDS)),
    "data_policy":"scikit-learn built-ins and deterministic synthetic data; no external downloads in primary benchmark",
    "artifacts":{}
}
for p in [RAW_PATH,ABLATION_PATH,PRUNING_PATH,TIMING_PATH,
          ARTIFACT_DIR/"summary_by_task.csv",ARTIFACT_DIR/"paired_rank_effects_holm.csv"]:
    if p.exists():
        manifest["artifacts"][p.name]={"bytes":p.stat().st_size,"sha256":sha256_file(p)}
with open(ARTIFACT_DIR/"reproducibility_manifest.json","w",encoding="utf-8") as f:
    json.dump(manifest,f,indent=2,default=str)
manifest

## Interpretation checklist before using these results in the paper

- [ ] All primary methods completed on all selected tasks/seeds/budgets; errors are understood and reported.
- [ ] SMAC3 is configured with the same legal conditional space and objective direction as the other methods.
- [ ] Optuna multivariate/group TPE is reported with its actual installed Optuna version; Optuna 5 changes defaults, so distinguish default sampler from explicitly configured variants.
- [ ] Deduplicated TPE counts unique model evaluations separately from sampler trials and cache hits.
- [ ] Outer test metrics are used for final generalization comparisons; inner CV is only for optimization.
- [ ] Search cost is shown as unique configs, fold-fits, and wall-clock seconds; do not use one cost measure as a substitute for another.
- [ ] Parallel comparisons state the worker allocation and scheduling granularity.
- [ ] Ablations change one component at a time and use matched seeds/folds.
- [ ] The pruning audit reports counterfactual recomputations and labels the false-prune quantity as a proxy, not a formal error probability.
- [ ] Task-level ranks and effect sizes are reported; raw classification and regression scores are not pooled.
- [ ] The 22-task scikit-learn suite is described accurately. YAHPO results must not be claimed unless an objective/fidelity adapter is implemented and evaluated fairly.
- [ ] Archive source code, notebook, raw JSONL files, CSV summaries and manifest with a permanent DOI.

**No results are generated by publishing this notebook.** It is a fully specified local experiment protocol; run it on your hardware and use the outputs—not expected outcomes—to update the paper.